# Colab: OpenAI + Hugging Face (2 sections)

One notebook, **two API calls**:

| Section | Who you talk to | Key | Shows on OpenAI usage? |
|---------|-----------------|-----|------------------------|
| 1 | OpenAI cloud | `OPENAI_API_KEY` | Yes |
| 2 | Hugging Face router → Together | `HF_TOKEN` | No |

Both sections use the same **OpenAI Python package**.  
Only `base_url`, key, and model change.

### Before you run

1. Left sidebar → **Secrets** (key icon)  
2. Add secrets (like a Colab `.env`):
   - Name: `OPENAI_API_KEY` → your OpenAI key  
   - Name: `HF_TOKEN` → your Hugging Face token  
3. Enable notebook access for both  
4. Run cells top → bottom  

Do **not** paste keys into cells.

## Setup — install + load both keys

- Install the OpenAI client once per runtime.  
- Read secrets from Colab Secrets into `os.environ` (same idea as `load_dotenv()` on your laptop).  
- Print only `True`/`False` — never print the actual keys.

### Code cell — install `openai`

```python
!pip -q install openai
```

| Piece | Meaning |
|-------|---------|
| `!` | Run a **shell** command inside Colab (not pure Python) |
| `pip install openai` | Install the OpenAI Python package in **this runtime** |
| `-q` | Quiet mode (less install noise) |

Colab runtimes are temporary. After disconnect / new runtime, you often need to install again.

In [ ]:
!pip -q install openai

### Code cell — load both secrets

```python
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))
```

| Line / piece | Meaning |
|--------------|---------|
| `import os` | Read/write environment variables |
| `from google.colab import userdata` | Colab helper to read **Secrets** |
| `from openai import OpenAI` | OpenAI-compatible client (used in both sections) |
| `userdata.get("OPENAI_API_KEY")` | Fetch OpenAI key from Secrets panel (by name) |
| `userdata.get("HF_TOKEN")` | Fetch HF token from Secrets panel |
| `os.environ["..."] = ...` | Put the value into the environment (like loading `.env`) |
| `bool(os.environ.get(...))` | Print `True`/`False` only — **never print the key itself** |

If either prints `False`, fix Secrets names / notebook access, then re-run this cell.

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))

## Section 1 — OpenAI model (OpenAI cloud)

### What this does

Calls a real **OpenAI** model (`gpt-4.1-nano`) using your OpenAI subscription.

```text
OpenAI SDK  →  api.openai.com  →  OpenAI model  →  reply
```

### Why it looks so short

```python
openai_client = OpenAI()
```

No `base_url` and no explicit key → defaults to OpenAI cloud and picks up `OPENAI_API_KEY` from the environment.

### Billing / usage

This call **does** appear on your **OpenAI** usage / billing page.

### Code cell — call OpenAI cloud

```python
openai_client = OpenAI()

openai_reply = openai_client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one short sentence."}
    ],
)

print("=== OpenAI ===")
print(openai_reply.choices[0].message.content)
```

| Line / piece | Meaning |
|--------------|---------|
| `OpenAI()` | Create client with **defaults** → OpenAI cloud + `OPENAI_API_KEY` from env |
| `chat.completions.create(...)` | Send a chat request |
| `model="gpt-4.1-nano"` | Which OpenAI model to use |
| `messages=[...]` | Conversation list (here: one user question) |
| `role: "user"` | Message is from the user |
| `choices[0].message.content` | Text reply from the first (only) choice |

This is the same pattern as your local OpenAI tests.

In [ ]:
openai_client = OpenAI()

openai_reply = openai_client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one short sentence."}
    ],
)

print("=== OpenAI ===")
print(openai_reply.choices[0].message.content)

## Section 2 — Hugging Face model (HF router + Together)

### What this does

Still uses the **OpenAI Python package**, but pointed at **Hugging Face**, not OpenAI cloud.

```text
OpenAI SDK  →  router.huggingface.co  →  Together runs Qwen  →  reply
```

| Setting | Value | Meaning |
|---------|-------|---------|
| `base_url` | `https://router.huggingface.co/v1` | Talk to HF router, not `api.openai.com` |
| `api_key` | `HF_TOKEN` | Authenticate to Hugging Face |
| `model` | `Qwen/Qwen2.5-7B-Instruct:together` | Hub model + provider routing |

### Key concept: why `:together` is on the model name

Same idea as your local HF test scripts.

- **HUB style** (`InferenceClient`): `provider="together"` + plain model name `Qwen/...`  
- **OpenAI-client style** (this section): no `provider=` arg → put provider on the model string as `:together`

```text
HUB:      provider="together"  +  model="Qwen/..."
OpenAI:   model="Qwen/...:together"
```

`:together` is **routing**, not part of the model’s real Hub name.

### Key concept: this does **not** appear in OpenAI usage

You imported `OpenAI`, but you are **not** calling OpenAI’s cloud.

| Piece | Points to |
|-------|-----------|
| `base_url=...huggingface...` | Hugging Face |
| `api_key=HF_TOKEN` | HF token (not `OPENAI_API_KEY`) |

So:

- **OpenAI usage page** → does **not** show this call  
- **Hugging Face** (and Together via HF) → where usage lives  

### Compare to Section 1

| | Section 1 | Section 2 |
|--|-----------|-----------|
| Package | `openai` | `openai` (same) |
| Destination | OpenAI cloud | HF router → Together |
| Key | `OPENAI_API_KEY` | `HF_TOKEN` |
| Model | `gpt-4.1-nano` | `Qwen/...:together` |
| OpenAI billing? | Yes | No |

### Code cell — call Hugging Face (via OpenAI SDK)

```python
hf_client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

hf_reply = hf_client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one short sentence."}
    ],
)

print("=== Hugging Face (via Together) ===")
print(hf_reply.choices[0].message.content)
```

| Line / piece | Meaning |
|--------------|---------|
| `OpenAI(...)` | Same package as Section 1 — different destination |
| `base_url="https://router.huggingface.co/v1"` | Send requests to **HF router**, not `api.openai.com` |
| `api_key=os.environ["HF_TOKEN"]` | Authenticate with **HF token** (not OpenAI key) |
| `model="Qwen/Qwen2.5-7B-Instruct:together"` | Hub model id + `:together` provider routing |
| `chat.completions.create(...)` | Same chat API shape as Section 1 |
| `messages=[...]` | Same question, for easy comparison |
| `choices[0].message.content` | Text reply |

Compare to Section 1: same `create(...)` call — only client settings + model string differ.

In [ ]:
hf_client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

hf_reply = hf_client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one short sentence."}
    ],
)

print("=== Hugging Face (via Together) ===")
print(hf_reply.choices[0].message.content)